# Scenario: Predict Credit Card Fraud using **SVM** and **Naive Bayes**



Credit card fraud happens when someone — a fraudster or a thief — uses your stolen credit card or the information from that card to make unauthorized purchases in your name or take out cash advances using your account.

### Problem Statement:

Credit card companies such as **Citibank**, **HSBC**, and **American Express** need to recognize fraudulent credit card transactions so that customers are not charged for items that they did not purchase.

### Aim:

In this demo, you have to build a classification model to identify fraudulent credit card transactions

### Dataset Description
The datasets contains transactions made by credit cards in September 2013 by european cardholders.

Presents transactions that occurred in two days, where we have **492** frauds out of **284,807** transactions.

- **Time** - Number of seconds elapsed between this transaction and the first transaction in the dataset
- **V1-V28** - Encrpted attributes (or columns) to protect user identities and sensitive features (v1-v28)
- **Amount** - Transaction Amount
- **Class** - **1** for fraudulent transactions, **0** otherwise

[**Click Here!**](https://www.dropbox.com/s/oey6vxdcc4fiemv/creditcard.csv?dl=0) to download the data set

In [ ]:
#Downloading the data set from Dropbox
#Please run this cell in Google Colab

!wget https://www.dropbox.com/s/oey6vxdcc4fiemv/creditcard.csv

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
# from pandas_profiling import ProfileReport
sns.set()


import warnings
warnings.filterwarnings("ignore")
import os


print('Libraries Imported')

Libraries Imported


In [ ]:
# # Downsample Majority
# from sklearn.utils import resample
# df = pd.read_csv('creditcard.csv')
# not_fraud = df[df.Class==0]
# fraud = df[df.Class==1]

# not_fraud_downsampled = resample(not_fraud,
#                                 replace = False, # sample without replacement
#                                 n_samples = 2315, # match minority n
#                                 random_state = 42) # reproducible results

# # Combine Minority and Downsampled Majority
# downsampled = pd.concat([not_fraud_downsampled, fraud])
# downsampled.to_csv('creditcard2.csv',index=False)

In [ ]:
df = pd.read_csv('creditcard2.csv')
print(df.shape)
df.head()

(2807, 31)


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,82450.0,1.314539,0.590643,-0.666593,0.716564,0.301978,-1.125467,0.388881,-0.288390,-0.132137,...,-0.170307,-0.429655,-0.141341,-0.200195,0.639491,0.399476,-0.034321,0.031692,0.76,0
1,50554.0,-0.798672,1.185093,0.904547,0.694584,0.219041,-0.319295,0.495236,0.139269,-0.760214,...,0.202287,0.578699,-0.092245,0.013723,-0.246466,-0.380057,-0.396030,-0.112901,4.18,0
2,55125.0,-0.391128,-0.245540,1.122074,-1.308725,-0.639891,0.008678,-0.701304,-0.027315,-2.628854,...,-0.133485,0.117403,-0.191748,-0.488642,-0.309774,0.008100,0.163716,0.239582,15.00,0
3,116572.0,-0.060302,1.065093,-0.987421,-0.029567,0.176376,-1.348539,0.775644,0.134843,-0.149734,...,0.355576,0.907570,-0.018454,-0.126269,-0.339923,-0.150285,-0.023634,0.042330,57.00,0
4,90434.0,1.848433,0.373364,0.269272,3.866438,0.088062,0.970447,-0.721945,0.235983,0.683491,...,0.103563,0.620954,0.197077,0.692392,-0.206530,-0.021328,-0.019823,-0.042682,0.00,0


In [ ]:
df.Class.value_counts()

0    2315
1     492
Name: Class, dtype: int64

### Data Manipulation

**RobustScaler:** Unlike the previous scalers, the
centering and scaling statistics of RobustScaler is based on percentiles and are therefore not influenced by a few number of very large marginal outliers. Consequently, the resulting range of the transformed feature values is larger than for the previous scalers and, more importantly, are approximately similar: for both features most of the transformed values lie in a [-2, 3] range

In [ ]:
df['Amount'].values.shape

(2807,)

In [ ]:
from sklearn.preprocessing import RobustScaler
rs = RobustScaler()

#Fit_Transform the scaled_amount and scaled_time columns in the data set and dropping the Original Time and Amount Column from the data set

df['scaled_amount'] = rs.fit_transform(df['Amount'].values.reshape(-1,1))
df['scaled_time'] = rs.fit_transform(df['Time'].values.reshape(-1,1))
df.drop(['Time', 'Amount'], axis=1, inplace=True) #Dropping the Original Time and Amount Column from the data set

In [ ]:
df.columns

Index(['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11',
       'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21',
       'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Class',
       'scaled_amount', 'scaled_time'],
      dtype='object')

In [ ]:
df = df[['scaled_amount', 'scaled_time','V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11',
       'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21',
       'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Class']]
df.head()

,scaled_amount,scaled_time,V1,V2,V3,V4,V5,V6,V7,V8,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Class
0,-0.249922,-0.018253,1.314539,0.590643,-0.666593,0.716564,0.301978,-1.125467,0.388881,-0.288390,...,-0.058040,-0.170307,-0.429655,-0.141341,-0.200195,0.639491,0.399476,-0.034321,0.031692,0
1,-0.207313,-0.387894,-0.798672,1.185093,0.904547,0.694584,0.219041,-0.319295,0.495236,0.139269,...,-0.081298,0.202287,0.578699,-0.092245,0.013723,-0.246466,-0.380057,-0.396030,-0.112901,0
2,-0.072510,-0.334921,-0.391128,-0.245540,1.122074,-1.308725,-0.639891,0.008678,-0.701304,-0.027315,...,0.065716,-0.133485,0.117403,-0.191748,-0.488642,-0.309774,0.008100,0.163716,0.239582,0
3,0.450757,0.377186,-0.060302,1.065093,-0.987421,-0.029567,0.176376,-1.348539,0.775644,0.134843,...,-0.169706,0.355576,0.907570,-0.018454,-0.126269,-0.339923,-0.150285,-0.023634,0.042330,0
4,-0.259391,0.074274,1.848433,0.373364,0.269272,3.866438,0.088062,0.970447,-0.721945,0.235983,...,-0.282777,0.103563,0.620954,0.197077,0.692392,-0.206530,-0.021328,-0.019823,-0.042682,0


In [ ]:
scaled_amount = df['scaled_amount']
scaled_time = df['scaled_time']
df.drop(['scaled_amount', 'scaled_time'], axis=1, inplace=True)
df.insert(0, 'scaled_amount', scaled_amount)
df.insert(0, 'scaled_time', scaled_time)
df.head()

,scaled_time,scaled_amount,V1,V2,V3,V4,V5,V6,V7,V8,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Class
0,-0.018253,-0.249922,1.314539,0.590643,-0.666593,0.716564,0.301978,-1.125467,0.388881,-0.288390,...,-0.058040,-0.170307,-0.429655,-0.141341,-0.200195,0.639491,0.399476,-0.034321,0.031692,0
1,-0.387894,-0.207313,-0.798672,1.185093,0.904547,0.694584,0.219041,-0.319295,0.495236,0.139269,...,-0.081298,0.202287,0.578699,-0.092245,0.013723,-0.246466,-0.380057,-0.396030,-0.112901,0
2,-0.334921,-0.072510,-0.391128,-0.245540,1.122074,-1.308725,-0.639891,0.008678,-0.701304,-0.027315,...,0.065716,-0.133485,0.117403,-0.191748,-0.488642,-0.309774,0.008100,0.163716,0.239582,0
3,0.377186,0.450757,-0.060302,1.065093,-0.987421,-0.029567,0.176376,-1.348539,0.775644,0.134843,...,-0.169706,0.355576,0.907570,-0.018454,-0.126269,-0.339923,-0.150285,-0.023634,0.042330,0
4,0.074274,-0.259391,1.848433,0.373364,0.269272,3.866438,0.088062,0.970447,-0.721945,0.235983,...,-0.282777,0.103563,0.620954,0.197077,0.692392,-0.206530,-0.021328,-0.019823,-0.042682,0


In [ ]:

from sklearn.model_selection import train_test_split as holdout #Importing the train_test_split from sklearn library
x = np.array(df.iloc[:, df.columns != 'Class']) #Predictors
y = np.array(df.iloc[:, df.columns == 'Class']) #Target Column
x_train, x_test, y_train, y_test = holdout(x, y, test_size=0.2, random_state=0) #Splitting the data set into training and testing set

### SVM

In [ ]:
from sklearn import svm
# Create and train a Linear SVM model
msvm = svm.LinearSVC(random_state=20)
msvm.fit(x_train, y_train)
predicted = msvm.predict(x_test)

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, predicted)

array([[450,   1],
       [ 19,  92]], dtype=int64)

### Naive Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

#Create and Train a Gaussian Classifier
gnb = GaussianNB()
gnb.fit(x_train, y_train)
y_pred = gnb.predict(x_test)

In [ ]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test,y_pred))

[[444   7]
 [ 19  92]]


### Model Optimization: GridSearchCV

**SVM Hyperparameters:**

- **Gamma**
 - Used with non-linear SVM. Commonly used non-linear kernel is the Radial Basis Function (RBF)
 - Gamma parameter of RBF controls the distance of the influence of a single training point
 - Low values of gamma indicate a large similarity radius which results in more points being grouped together
 - For high values of gamma, the points need to be very close to each other in order to be considered in the same group (or class)
 - Models with very large gamma values tend to overfit.

- **C**
 - Adds a penalty for each misclassified data point
 - If c is small, the penalty for misclassified points is low so a decision boundary with a large margin is chosen at the expense of a greater number of misclassifications
 - If c is large, SVM tries to minimize the number of misclassified examples due to high penalty which results in a decision boundary with a smaller margin
 - Penalty is not same for all misclassified examples
 - It is directly proportional to the distance to decision boundary

**NOTE:** If you want to learn more about SVM Hyper-parameters, [**Click Here!**](http://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html)

In [ ]:
#GridSearchCV
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# defining parameter range
# param_grid = {'C': [0.1, 1, 10, 100, 1000],'gamma': [1, 0.1, 0.01, 0.001, 0.0001], 'kernel': ['rbf']}
param_grid = {'C': [0.1, 1, 10],'gamma': ['scale', 'auto'], 'kernel': ['rbf']}

svc_grid = GridSearchCV(SVC(), param_grid, refit = True, verbose = 3, cv=2)

svc_grid.fit(x_train, y_train)


Fitting 2 folds for each of 6 candidates, totalling 12 fits
[CV 1/2] END ....C=0.1, gamma=scale, kernel=rbf;, score=0.964 total time=   0.0s
[CV 2/2] END ....C=0.1, gamma=scale, kernel=rbf;, score=0.966 total time=   0.0s
[CV 1/2] END .....C=0.1, gamma=auto, kernel=rbf;, score=0.965 total time=   0.0s
[CV 2/2] END .....C=0.1, gamma=auto, kernel=rbf;, score=0.960 total time=   0.0s
[CV 1/2] END ......C=1, gamma=scale, kernel=rbf;, score=0.970 total time=   0.0s
[CV 2/2] END ......C=1, gamma=scale, kernel=rbf;, score=0.973 total time=   0.0s
[CV 1/2] END .......C=1, gamma=auto, kernel=rbf;, score=0.965 total time=   0.0s
[CV 2/2] END .......C=1, gamma=auto, kernel=rbf;, score=0.972 total time=   0.0s
[CV 1/2] END .....C=10, gamma=scale, kernel=rbf;, score=0.970 total time=   0.0s
[CV 2/2] END .....C=10, gamma=scale, kernel=rbf;, score=0.980 total time=   0.0s
[CV 1/2] END ......C=10, gamma=auto, kernel=rbf;, score=0.972 total time=   0.0s
[CV 2/2] END ......C=10, gamma=auto, kernel=rbf;,

GridSearchCV(cv=2, estimator=SVC(),
             param_grid={'C': [0.1, 1, 10], 'gamma': ['scale', 'auto'],
                         'kernel': ['rbf']},
             verbose=3)

In [ ]:
print('Best Parameters:',svc_grid.best_params_, 'Best Estimator:',svc_grid.best_estimator_)

Best Parameters: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'} Best Estimator: SVC(C=10)


In [ ]:
grid_predictions = svc_grid.predict(x_test)
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test,grid_predictions))
#print(classification_report(y_test, grid_predictions))

[[448   3]
 [ 17  94]]
